In [21]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy": DATA_RAW / "enrichment-success-andy.json",
    "dishita": DATA_RAW / "enrichment-success-dish.json",
    "riya": DATA_RAW / "enrichment-success-riya.json",
    "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

USERS = list(USER_FILES.keys())

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]
N_MOOD_CLUSTERS = 4
TOP_N_CONTENT = 300
TOP_N_TAG = 200
TOP_N_NEIGHBOR = 150
FINAL_CANDIDATE_K = 1000
TOP_K_TAGS = 10
MIN_NEIGHBOR_MS = 30_000
MAX_NEIGHBOR_SKIP = 0.3

['andy', 'dishita', 'riya', 'priyanka']


In [ ]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track.
Tries all_tags first, falls back to sc_genres, then lastfm_tags.
If none are available, returns an empty list.
"""
def resolve_tags(row) -> list:
    """Canonical tag: all_tags → sc_genres → lastfm_tags."""
    for col in ("all_tags", "sc_genres", "lastfm_tags"):
        val = row.get(col)
        if isinstance(val, list) and len(val) > 0:
            return [t.strip().lower() for t in val]
    return []

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df.
Filters out non-track plays (episodes, audiobooks), parses timestamps, and
adds flag for plays (>=30 seconds) that were not skipped.
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)
    df = pd.json_normalize(raw)
    df['user'] = user

    af_rename = {f"audio_features.{feat}": feat for feat in AUDIO_FEATURES}
    df = df.rename(columns=af_rename)

    # Keep only track plays
    df = df[df["spotify_track_uri"].notna() & df["episode_name"].isna()].copy()

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"] = df["skipped"].fillna(False).astype(bool)
    df["tags"] = df.apply(resolve_tags, axis=1)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= 30_000)
    return df

"""
Loads and combines listening history for all users into a single df.
Each row represents one play event, tagged with the user it belongs to.
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = [load_user_events(p, u) for u, p in user_files.items()]
    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 29,887
Unique tracks: 7,068
Users: <StringArray>
['andy', 'dishita', 'riya', 'priyanka']
Length: 4, dtype: str


In [38]:
# TRACK TABLE (ITEM VECTORS)

# TRACK TABLE
track_table = (
    events[[
        "spotify_track_uri",
        "master_metadata_track_name",
        "master_metadata_album_artist_name",
        "master_metadata_album_album_name",
        "tags",
        *AUDIO_FEATURES,
    ]]
    .drop_duplicates(subset="spotify_track_uri")
    .reset_index(drop=True)
    .rename(columns={
        "master_metadata_track_name": "track_name",
        "master_metadata_album_artist_name": "artist_name",
        "master_metadata_album_album_name": "album_name",
    })
)

track_table["track_id"] = track_table.index

uri_to_id = dict(zip(track_table["spotify_track_uri"], track_table["track_id"]))
id_to_uri = dict(zip(track_table["track_id"], track_table["spotify_track_uri"]))

# NORMALIZED AUDIO FEATURE MATRIX
audio_matrix = track_table[AUDIO_FEATURES].fillna(0).values.astype(float)
audio_scaler = MinMaxScaler()
audio_matrix_norm = audio_scaler.fit_transform(audio_matrix)

# TAG TF-IDF MATRIX
tag_docs = track_table["tags"].apply(lambda t: " ".join(t) if t else "")
tfidf = TfidfVectorizer(min_df=2, max_features=500)
tag_matrix = tfidf.fit_transform(tag_docs)
tag_vocab = tfidf.get_feature_names_out()

# ARTIST ID
track_table["artist_id"] = pd.Categorical(track_table["artist_name"]).codes

# SAVE TRACK TABLE & FITTED MODELS
track_table.to_csv(DATA_PROC / "track_features.csv", index=False)
joblib.dump(audio_scaler, MODELS_DIR / "audio_scaler.joblib")
joblib.dump(tfidf, MODELS_DIR / "tfidf.joblib")

print(f"Track table saved: {len(track_table):,} tracks, tag vocab = {len(tag_vocab)}")
print(f"Tag vocab size: {len(tag_vocab)}")

Track table saved: 7,068 tracks, tag vocab = 330
Tag vocab size: 330


In [51]:
# USER PROFILES

# PER-(USER, TRACK) PLAY-STATS
events["track_id"] = events["spotify_track_uri"].map(uri_to_id)

play_stats = (
    events.groupby(["user", "track_id"])
    .agg(
        n_plays = ("ms_played", "count"),
        avg_ms = ("ms_played", "mean"),
        skip_count = ("skipped", "sum"),
        meaningful_pct = ("meaningful", "mean"),
    )
    .reset_index()
)

play_stats["skip_rate"] = play_stats["skip_count"] / play_stats["n_plays"]

# PLAY WEIGHT
"""
Assigns a weight to a (user, track) play record showing how much the user likes the track.
Tracks that are frequently skipped early get a weight of 0.
Otherwise, weight increases with the percentage of meaningful listens and 
the log of total play count (to dampen the reduced impact of excessive replays).
"""
def play_weight(row) -> float:
    if row["skip_rate"] > 0.7 and row["avg_ms"] < 15_000:
        return 0.0
    return row["meaningful_pct"] * np.log1p(row["n_plays"])

play_stats["weight"] = play_stats.apply(play_weight, axis=1)
play_stats.to_csv(DATA_PROC / "play_stats.csv", index=False)

# LONG-TERM TASTE PROFILES
"""
Builds a single vector representing a user's overall audio taste by computing a weighted average
of all tracks they've listened to.
Tracks with higher engagement (more meaningful plays, more listens) contribute more to the profile.
Returns a 12-dimensional vector matching the audio feature space.
"""
def build_long_term_profile(user: str) -> np.ndarray:
    u = play_stats[(play_stats["user"] == user) & (play_stats["weight"] > 0)]
    if u.empty:
        return np.zeros(audio_matrix_norm.shape[1])
    return np.average(
        audio_matrix_norm[u["track_id"].values], 
        axis=0, 
        weights=u["weight"].values
    )

long_term_profiles = {u: build_long_term_profile(u) for u in USERS}

# SAVE: ONE ROW PER USER, ONE COLUMN PER AUDIO FEATURE
pd.DataFrame(long_term_profiles, index=AUDIO_FEATURES).T \
    .rename_axis("user") \
    .reset_index() \
    .to_csv(DATA_PROC / "user_profiles.csv", index=False)

# TAG DISTRIBUTIONS PER USER
"""
Builds a weighted tag profile for a user by averaging the TF-IDF tag vectors of all tracks they've
listened to.
Captures which genres and descriptors are more characteristic of their taste.
Used for tag-based candidate expansion and user-user similarity.
"""
def build_user_tag_dist(user: str) -> np.ndarray:
    u = play_stats[(play_stats["user"] == user) & (play_stats["weight"] > 0)]
    if u.empty:
        return np.zeros(tag_matrix.shape[1])
    idx, w = u["track_id"].values, u["weight"].values
    w_sum = np.array(tag_matrix[idx].multiply(w[:, None]).sum(axis=0)).flatten()
    return w_sum / (w.sum() + 1e-9)

user_tag_profiles = {u: build_user_tag_dist(u) for u in USERS}

# SAVE: ONE ROW PER USER, ONE COLUMN PER TAG
pd.DataFrame(user_tag_profiles, index=tag_vocab).T \
    .rename_axis("user") \
    .reset_index() \
    .to_csv(DATA_PROC / "user_tag_profiles.csv", index=False)

# MOOD CLUSTERS PER USER
mood_feat_idx = [AUDIO_FEATURES.index(f) for f in MOOD_FEATURES]
mood_scaler = MinMaxScaler()
mood_matrix_norm = mood_scaler.fit_transform(audio_matrix[:, mood_feat_idx])

def build_mood_clusters(user: str):
    u = play_stats[(play_stats["user"] == user) & (play_stats["weight"] > 0)]
    idx, w = u["track_id"].values, u["weight"].values
    X = mood_matrix_norm[idx]

    k = min(N_MOOD_CLUSTERS, len(idx))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X, sample_weight=w)

    tag_dist = {}
    for c in range(k):
        mask = labels == c
        c_idx, c_w = idx[mask], w[mask]
        if len(c_idx) == 0:
            tag_dist[c] = np.zeros(tag_matrix.shape[1])
        else:
            ws = np.array(tag_matrix[c_idx].multiply(c_w[:, None]).sum(axis=0)).flatten()
            tag_dist[c] = ws / (c_w.sum() + 1e-9)
    return km.cluster_centers_, labels, idx, tag_dist

mood_clusters = {u: build_mood_clusters(u) for u in USERS}

# SAVE ONE CSV PER USER WITH CENTROIDS ROWS LABELLED BY CLUSTER INDEX
for u in USERS:
    centroids, *_ = mood_clusters[u]
    pd.DataFrame(centroids, columns=MOOD_FEATURES) \
        .rename_axis("cluster") \
        .reset_index() \
        .assign(user=u) \
        .to_csv(DATA_PROC / f"mood_clusters_{u}.csv", index=False)
    
joblib.dump(mood_scaler, MODELS_DIR / "mood_scaler.joblib")

# USER-USER SIMILARITY
user_sim_df = pd.DataFrame(
    cosine_similarity(np.stack([user_tag_profiles[u] for u in USERS])),
    index=USERS, columns=USERS
)

print("User profiles saved -> data/processed/")
print("mood_scaler.joblib -> models/")
print(user_sim_df.round(3))

User profiles saved -> data/processed/
mood_scaler.joblib -> models/
           andy  dishita   riya  priyanka
andy      1.000    0.835  0.534     0.906
dishita   0.835    1.000  0.628     0.818
riya      0.534    0.628  1.000     0.741
priyanka  0.906    0.818  0.741     1.000


In [26]:
# SCORING

# RE-RANKING

# UI-WIDGETS (?)

# TESTING

# DEMO (?)